In [1]:
print("Jay Ganesh")

Jay Ganesh


### Data Transformation

In [2]:
import os

In [3]:
%pwd

'd:\\ProjectAarya\\MLOPs\\Code\\Ch21 NLP project with hugging face and transformer\\TextSummarization\\research'

In [ ]:
# os.chdir('..')
# %pwd

'd:\\ProjectAarya\\MLOPs\\Code\\Ch21 NLP project with hugging face and transformer\\TextSummarization'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
  root_dir: Path
  data_path: Path
  tokenizer_name: Path

In [6]:
from src.TextSummarizer.constants import *
from src.TextSummarizer.utils.common import read_yaml, create_directories


In [7]:
class ConfigurationManager:
    def __init__(self,config_path=CONFIG_FILE_PATH,params_path=PARAM_FILE_PATH):
        self.config=read_yaml(config_path)
        self.params=read_yaml(params_path)
        
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self)-> DataTransformationConfig:
        config=self.config.data_transformation
        create_directories([config.root_dir])
        
        data_transformation_config=DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name
        )
        return data_transformation_config

In [8]:
import os
from src.TextSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk

## Data Transformation Component

In [14]:
class DataTransformation:
    def __init__(self,config:DataTransformationConfig):
        self.config=config
        self.tokenizer=AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self,example_batch):
        input_encodings=self.tokenizer(example_batch['dialogue'],max_length=1024,truncation=True)
        target_encodings=self.tokenizer(example_batch['summary'],max_length=1024,truncation=True)
        return{
            "input_ids":input_encodings["input_ids"],
            "attention_mask": input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    
    def convert(self):
        data_set=load_from_disk(self.config.data_path)
        dataset_samsum_pt=data_set.map(self.convert_examples_to_features,batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))
    

In [15]:
config=ConfigurationManager()
data_transformer_config=config.get_data_transformation_config()
data_transformer=DataTransformation(data_transformer_config)
data_transformer.convert()

[2026-04-08 13:06:12,745]:INFO common yaml file: config\config.yaml loaded and returned successfully
[2026-04-08 13:06:12,751]:INFO common yaml file: params.yaml loaded and returned successfully
[2026-04-08 13:06:12,757]:INFO common artifacts created successfully
[2026-04-08 13:06:12,760]:INFO common artifacts/data_transformation created successfully
[2026-04-08 13:06:24,867]:INFO _client HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-08 13:06:24,915]:INFO _client HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-04-08 13:06:25,066]:INFO _client HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"


d:\ProjectAarya\MLOPs\Code\Ch21 NLP project with hugging face and transformer\TextSummarization\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jay\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[2026-04-08 13:06:27,841]:INFO _client HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-08 13:06:27,900]:INFO _client HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"
[2026-04-08 13:06:27,980]:INFO _client HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"
[2026-04-08 13:08:28,446]:INFO _client HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 504 Gateway Time-out"
[2026-04-08 13:08:28,876]:INFO _client HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=true&expand=false "HTTP/1.1 200 O

[2026-04-08 13:08:30,714]:WARNING _http Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-04-08 13:08:31,020]:INFO _client HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-08 13:08:31,065]:INFO _client HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/special_tokens_map.json "HTTP/1.1 200 OK"
[2026-04-08 13:08:31,135]:INFO _client HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/special_tokens_map.json "HTTP/1.1 200 OK"
[2026-04-08 13:08:31,532]:INFO _client HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 16401.78 examples/s]


In [13]:
print(data_transformer_config)

DataTransformationConfig(root_dir='artifacts/data_transformation', data_path='artifacts/data_ingestion/samsum_dataset', tokenizer_name='google/pegasus-cnn_dailymail')
